In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split
)

from sklearn.preprocessing import (
    StandardScaler
)

from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.ensemble import (
    RandomForestRegressor
)

# =========================================
# LOAD ORIGINAL TEST IDS
# =========================================

original_test = pd.read_csv(
    "../data/raw/test.csv"
)

test_ids = original_test["Id"]

# =========================================
# LOAD DATA
# =========================================

train_df = pd.read_csv(
    "../data/processed/featured_train.csv"
)

test_df = pd.read_csv(
    "../data/processed/featured_test.csv"
)
# =========================================
# LOG TRANSFORM TARGET
# =========================================

train_df["SalePrice"] = np.log1p(
    train_df["SalePrice"]
)
# =========================================
# SPLIT FEATURES AND TARGET
# =========================================

X = train_df.drop(
    "SalePrice",
    axis=1
)

y = train_df["SalePrice"]
# =========================================
# ONE HOT ENCODING
# =========================================

full_data = pd.concat(
    [X, test_df]
)

full_data = pd.get_dummies(
    full_data,
    drop_first=True
)

X = full_data.iloc[:len(X), :]
X_test_final = full_data.iloc[len(X):, :]
# =========================================
# TRAIN TEST SPLIT
# =========================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
# =========================================
# FEATURE SCALING
# =========================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_valid_scaled = scaler.transform(
    X_valid
)
# =========================================
# MODEL EVALUATION FUNCTION
# =========================================

def evaluate_model(
    name,
    model,
    X_test,
    y_test
):

    predictions = model.predict(X_test)

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        predictions
    )

    print(f"\n{name}")
    print("-" * 30)
    print("MSE :", mse)
    print("RMSE:", rmse)
    print("R2  :", r2)
# =========================================
# LINEAR REGRESSION
# =========================================

lr_model = LinearRegression()

lr_model.fit(
    X_train_scaled,
    y_train
)

evaluate_model(
    "Linear Regression",
    lr_model,
    X_valid_scaled,
    y_valid
)
# =========================================
# RIDGE REGRESSION
# =========================================

ridge_model = Ridge(alpha=10)

ridge_model.fit(
    X_train_scaled,
    y_train
)

evaluate_model(
    "Ridge Regression",
    ridge_model,
    X_valid_scaled,
    y_valid
)
# =========================================
# LASSO REGRESSION
# =========================================

lasso_model = Lasso(alpha=0.001)

lasso_model.fit(
    X_train_scaled,
    y_train
)

evaluate_model(
    "Lasso Regression",
    lasso_model,
    X_valid_scaled,
    y_valid
)
# =========================================
# ELASTIC NET
# =========================================

elastic_model = ElasticNet(
    alpha=0.001,
    l1_ratio=0.5
)

elastic_model.fit(
    X_train_scaled,
    y_train
)

evaluate_model(
    "ElasticNet",
    elastic_model,
    X_valid_scaled,
    y_valid
)
# =========================================
# RANDOM FOREST
# =========================================

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

evaluate_model(
    "Random Forest",
    rf_model,
    X_valid,
    y_valid
)
# =========================================
# FEATURE IMPORTANCE
# =========================================

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df.head(15)
# =========================================
# FINAL TEST PREDICTIONS
# =========================================

final_predictions = rf_model.predict(
    X_test_final
)

final_predictions = np.expm1(
    final_predictions
)
# =========================================
# CREATE SUBMISSION FILE
# =========================================

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": final_predictions
})

submission.to_csv(
    "../outputs/predictions/submission.csv",
    index=False
)

print("Submission file saved.")



Linear Regression
------------------------------
MSE : 0.018009534558173723
RMSE: 0.13419960714612292
R2  : 0.8906542360512986

Ridge Regression
------------------------------
MSE : 0.015345747175039002
RMSE: 0.12387795273994079
R2  : 0.9068275505511773

Lasso Regression
------------------------------
MSE : 0.012442130002763976
RMSE: 0.11154429614625741
R2  : 0.9244570032664271

ElasticNet
------------------------------
MSE : 0.013594358587990501
RMSE: 0.11659484803365242
R2  : 0.9174611914375231

Random Forest
------------------------------
MSE : 0.018721697021384615
RMSE: 0.13682725248058084
R2  : 0.8863303070600276
Submission file saved.
